# Customer Churn — Baseline ML Notebook

**Purpose:** educational baseline only. Dataset has 12 records, so evaluation is not reliable for real deployment.

Run from this folder with `customer-churn-training.csv` beside the notebook.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [ ]:
df = pd.read_csv('customer-churn-training.csv')
df.head()

In [ ]:
print('Shape:', df.shape)
print('\nMissing values:\n', df.isna().sum())
print('\nLabel counts:\n', df['churned'].value_counts())
print('\nBasic statistics:\n', df.describe(include='all'))

## Define target and features

Drop `customer_id` because it is an identifier. The feature/label timing is not documented; leakage cannot be ruled out until the data provider confirms the snapshot date and churn horizon.

In [ ]:
X = df.drop(columns=['customer_id', 'churned'])
y = df['churned']

numeric_features = ['tenure_months', 'support_tickets', 'monthly_spend_inr', 'last_login_days']
categorical_features = ['plan_type']
preprocess = ColumnTransformer([
    ('numeric', StandardScaler(), numeric_features),
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

## Train/test split

A small random holdout is shown for practice. With only 12 records, the test set is tiny and results vary substantially. Treat all metrics as a demonstration—not evidence of generalization.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Train labels:', y_train.value_counts().to_dict())
print('Test labels:', y_test.value_counts().to_dict())

## Majority-class baseline and logistic regression

In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)

model = Pipeline([
    ('preprocess', preprocess),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])
model.fit(X_train, y_train)

pred_dummy = dummy.predict(X_test)
pred_model = model.predict(X_test)

In [ ]:
def report(name, y_true, y_pred):
    print(f'--- {name} ---')
    print('Accuracy:', round(accuracy_score(y_true, y_pred), 3))
    print('Precision:', round(precision_score(y_true, y_pred, zero_division=0), 3))
    print('Recall:', round(recall_score(y_true, y_pred, zero_division=0), 3))
    print('F1:', round(f1_score(y_true, y_pred, zero_division=0), 3))
    print('Confusion matrix:\n', confusion_matrix(y_true, y_pred))

report('Majority baseline', y_test, pred_dummy)
report('Logistic regression', y_test, pred_model)

## Example prediction scores

These scores are not calibrated probabilities and should not be used to make decisions with this dataset.

In [ ]:
model.predict_proba(X_test)

## Responsible use checklist

- [ ] Confirm data source, lawful basis, consent/notice, and retention.
- [ ] Define prediction snapshot and future churn window.
- [ ] Validate on a larger, time-separated, representative dataset.
- [ ] Compare to a simple non-ML baseline.
- [ ] Review false-positive and false-negative costs with stakeholders.
- [ ] Keep retention outreach human-reviewed; permit abstention.
- [ ] Monitor drift, data quality, and appropriate fairness metrics.
- [ ] Document rollback to manual process.

**Conclusion:** This notebook is a reproducible learning baseline only. The 12-row sample is not adequate for production decisions.